In [1]:
%pip install git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3
%pip install accelerate


  Cloning https://github.com/huggingface/transformers (to revision v4.49.0-Gemma-3) to /private/var/folders/sk/72429_w95sz049jdbs5_8p3n68w8h9/T/pip-req-build-4puyvc05
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /private/var/folders/sk/72429_w95sz049jdbs5_8p3n68w8h9/T/pip-req-build-4puyvc05
  Running command git checkout -q 1c0f782fe5f983727ff245c4c1b3906f9b99eec2
  Resolved https://github.com/huggingface/transformers to commit 1c0f782fe5f983727ff245c4c1b3906f9b99eec2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import time

# MPS 장치 사용 가능 여부 확인
if not torch.backends.mps.is_available():
    print("MPS 장치를 사용할 수 없습니다.")
    exit()

# 모델 및 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it") # 4B 모델은 메모리 이슈로 2B모델로 대체
model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it").to("mps")

# 입력 텍스트
input_text = "What is the capital of France?"

# 입력 텍스트 토큰화
input_ids = tokenizer.encode(input_text, return_tensors="pt").to("mps")

# 추론 시간 측정
num_iterations = 100
total_time = 0

with torch.no_grad():
    for _ in range(num_iterations):
        start_time = time.time()
        output = model.generate(input_ids, max_length=50)
        torch.mps.synchronize()  # MPS 동기화
        total_time += time.time() - start_time

average_time = total_time / num_iterations
print(f"평균 추론 시간: {average_time:.6f} 초")

# 출력 결과 디코딩
output_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"출력 결과: {output_text}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

평균 추론 시간: 0.986522 초
출력 결과: What is the capital of France?

The capital of France is Paris. It is the political, economic, and cultural center of the country.


In [ ]:
# pip install accelerate

from transformers import AutoProcessor, Gemma3ForConditionalGeneration
from PIL import Image
import requests
import torch

model_id = "google/gemma-3-4b-it"

model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id, device_map="auto"
).eval()

processor = AutoProcessor.from_pretrained(model_id)

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful assistant."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg"},
            {"type": "text", "text": "Describe this image in detail."}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages, 
    add_generation_prompt=True, 
    tokenize=True,
    return_dict=True, 
    return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)

# **Overall Impression:** The image is a close-up shot of a vibrant garden scene, 
# focusing on a cluster of pink cosmos flowers and a busy bumblebee. 
# It has a slightly soft, natural feel, likely captured in daylight.


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Okay, here's a detailed description of the image:

**Overall Impression:**

The image is a close-up shot of a vibrant garden scene, focusing on a pink cosmos flower with a bee actively visiting it. The composition is natural and slightly blurred in the background, creating a soft, inviting feel.

**Main Subject - The Cosmos Flower:**

*   **Color:** The dominant color is a lovely, soft pink. The petals have a slightly creamy or blush tone.
*
